**Akarsh Dubey CS25MTECH14001**

**Atish Kadam CS25MTECH14003**

# Assignement - 1

Build Graph Classification Model using
1. GCN (Graph Convolution Network)
2. GAT (Graph Attention Network)
3. GraphSage

**Uploading File**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving edges.csv to edges.csv
Saving features.csv to features.csv
Saving labels.csv to labels.csv


**Importing requirements**

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=False)
print("Dependencies ready.")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

from sklearn.model_selection import train_test_split as split_data
from sklearn.preprocessing import StandardScaler as Scaler
from sklearn.utils.class_weight import compute_class_weight as class_weights
from sklearn.metrics import precision_score as prec, recall_score as rec

Dependencies ready.


In [ ]:
# Reproducibility
RANDOM_STATE = 42

def set_seed(seed_value):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)

set_seed(RANDOM_STATE)

# Device Setup
def get_compute_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

compute_device = get_compute_device()

print("Running on:", compute_device)
print("Torch version:", torch.__version__)

Running on: cpu
Torch version: 2.10.0+cpu


**Dataset Preparation and Cleaning**

In [ ]:
# Load CSV Files
edge_data   = pd.read_csv("edges.csv")
label_data  = pd.read_csv("labels.csv")
feature_data = pd.read_csv("features.csv")


#Clean Column Names
def clean_columns(df_list):
    for frame in df_list:
        frame.columns = frame.columns.str.strip()

clean_columns([edge_data, label_data, feature_data])


# Identify Node ID Column
def detect_id_column(frame):
    possible_names = ["vertex", "node", "node_id", "id"]
    for col in frame.columns:
        if col.lower() in possible_names:
            return col
    return None

feat_id_col  = detect_id_column(feature_data)
label_id_col = detect_id_column(label_data)


#Align Ordering
def sort_if_possible(frame, col_name):
    if col_name is not None:
        return frame.sort_values(col_name).reset_index(drop=True)
    return frame

feature_data = sort_if_possible(feature_data, feat_id_col)
label_data   = sort_if_possible(label_data, label_id_col)


#Extract Features & Labels
excluded_cols = {feat_id_col, "Vertex", "vertex", "node", "node_id", "id"}

feature_columns = [col for col in feature_data.columns if col not in excluded_cols]

X_raw = feature_data[feature_columns].to_numpy(dtype=np.float32)
y_np  = label_data["label"].to_numpy()

num_nodes = len(y_np)


# Edge Processing
edge_columns = edge_data.columns.tolist()
source_nodes = edge_data[edge_columns[0]].to_numpy()
target_nodes = edge_data[edge_columns[1]].to_numpy()


#Node Index Mapping
if label_id_col is None:
    valid_nodes = set(range(num_nodes))
    id_to_index = {i: i for i in range(num_nodes)}
else:
    valid_nodes = set(label_data[label_id_col].to_numpy())
    id_to_index = {node_id: idx for idx, node_id in enumerate(label_data[label_id_col].to_numpy())}


# Filter Valid Edges
valid_mask = np.array([
    (s in valid_nodes and t in valid_nodes)
    for s, t in zip(source_nodes, target_nodes)
])

filtered_src = np.array([id_to_index[s] for s in source_nodes[valid_mask]])
filtered_dst = np.array([id_to_index[t] for t in target_nodes[valid_mask]])


# Printing Dataset Summary
print(f"Total Nodes   : {num_nodes}")
print(f"Total Edges   : {len(filtered_src)}")
print(f"Feature Count : {X_raw.shape[1]}")
print(f"Label Dist.   : {dict(zip(*np.unique(y_np, return_counts=True)))}")

Total Nodes   : 2000
Total Edges   : 8000
Feature Count : 16
Label Dist.   : {np.int64(0): np.int64(1000), np.int64(1): np.int64(1000)}


**Spliting Dataset**

In [ ]:
total_samples = X_raw.shape[0]

if total_samples != len(y_np):
    min_count = min(total_samples, len(y_np))
    X_raw = X_raw[:min_count]
    y_np  = y_np[:min_count]
    total_samples = min_count

node_indices = np.arange(total_samples)

# Dataset Split
train_nodes, temp_nodes = split_data(
    node_indices,
    test_size=0.40,
    random_state=RANDOM_STATE,
    stratify=y_np
)

val_nodes, test_nodes = split_data(
    temp_nodes,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_np[temp_nodes]
)

**Feature Normaliastion**

In [ ]:
#Feature Normalization
scaler_instance = Scaler()
scaler_instance.fit(X_raw[train_nodes])

X_norm = scaler_instance.transform(X_raw)

# Edge Processing (Undirected Graph)

edges_combined = np.stack([
    np.concatenate((filtered_src, filtered_dst)),
    np.concatenate((filtered_dst, filtered_src))
], axis=0)

edge_idx_tensor = torch.tensor(edges_combined, dtype=torch.long)

# Convert to Tensors
x_tensor = torch.tensor(X_norm, dtype=torch.float32)
y_tensor = torch.tensor(y_np,  dtype=torch.long)

# Mask Utility
def build_mask(idxs, length):
    mask_vec = torch.zeros(length, dtype=torch.bool)
    mask_vec[idxs] = True
    return mask_vec

train_mask = build_mask(train_nodes, total_samples)
val_mask   = build_mask(val_nodes, total_samples)
test_mask  = build_mask(test_nodes, total_samples)


# Create Graph Object
graph_data = Data(
    x=x_tensor,
    edge_index=edge_idx_tensor,
    y=y_tensor,
    train_mask=train_mask,
    val_mask=val_mask,
    test_mask=test_mask
).to(compute_device)

train_classes = np.unique(y_np[train_nodes])

cw_values = class_weights(
    class_weight="balanced",
    classes=train_classes,
    y=y_np[train_nodes]
)

cw_tensor = torch.tensor(cw_values, dtype=torch.float32).to(compute_device)

print(
    f"Train: {train_mask.sum().item()} | "
    f"Val: {val_mask.sum().item()} | "
    f"Test: {test_mask.sum().item()}"
)

print(
    f"Input Features: {graph_data.num_node_features} | "
    f"Classes: {y_tensor.unique().numel()}"
)

Train: 1200 | Val: 400 | Test: 400
Input Features: 16 | Classes: 2


# Implementation of GCN

**Normalisation Utility**

In [ ]:
def compute_gcn_factors(total_nodes, edges, device):
    identity_idx = torch.arange(total_nodes, device=device)
    self_connections = torch.stack([identity_idx, identity_idx], dim=0)

    expanded_edges = torch.cat((edges, self_connections), dim=1)

    degree_vals = torch.zeros(total_nodes, device=device)
    degree_vals.scatter_add_(
        0,
        expanded_edges[1],
        torch.ones(expanded_edges.size(1), device=device)
    )

    # normalization term: (d_i * d_j)^(-1/2)
    norm_factors = (degree_vals[expanded_edges[0]] * degree_vals[expanded_edges[1]])
    norm_factors = norm_factors.clamp(min=1e-9).pow(-0.5)

    return expanded_edges, norm_factors

**GCN Single Layer**

In [ ]:
class GraphConvLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.linear_map = nn.Linear(input_dim, output_dim, bias=False)
        nn.init.xavier_uniform_(self.linear_map.weight)

    def forward(self, features, edge_idx, edge_weights):
        src_nodes, dst_nodes = edge_idx
        num_nodes, feat_dim  = features.size()

        aggregated = torch.zeros(num_nodes, feat_dim, device=features.device)

        aggregated.scatter_add_(
            0,
            dst_nodes.view(-1, 1).expand(-1, feat_dim),
            edge_weights.view(-1, 1) * features[src_nodes]
        )

        transformed = self.linear_map(aggregated)

        return F.normalize(transformed, p=2, dim=1)

**Entire GCN Model**

In [ ]:
# Full GCN Model
class NodeGCN(nn.Module):
    """
    Multi-layer GCN with normalization + dropout.
    """
    def __init__(self, input_dim, hidden_dim, output_dim, drop_prob=0.5):
        super().__init__()

        self.conv1 = GraphConvLayer(input_dim, hidden_dim)
        self.conv2 = GraphConvLayer(hidden_dim, hidden_dim)
        self.conv3 = GraphConvLayer(hidden_dim, output_dim)

        self.norm1 = nn.BatchNorm1d(hidden_dim)
        self.norm2 = nn.BatchNorm1d(hidden_dim)

        self.drop_rate = drop_prob

    def forward(self, x, edge_idx, norm_vals):
        h = self.conv1(x, edge_idx, norm_vals)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.drop_rate, training=self.training)

        h = self.conv2(h, edge_idx, norm_vals)
        h = self.norm2(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.drop_rate, training=self.training)

        logits = self.conv3(h, edge_idx, norm_vals)
        return logits

gcn_edges, gcn_weights = compute_gcn_factors(
    graph_data.num_nodes,
    graph_data.edge_index,
    compute_device
)

# GAT Model (Everything From Scratch)

In [ ]:
# Multi-Head Attention Layer
class AttentionLayer(nn.Module):
    def __init__(self, input_dim, output_dim, num_heads=4, drop_rate=0.5, merge_heads=True):
        super().__init__()

        self.num_heads = num_heads
        self.out_dim   = output_dim
        self.merge     = merge_heads
        self.drop_p    = drop_rate

        # learnable parameters
        self.weight = nn.Parameter(torch.empty(num_heads, input_dim, output_dim))
        self.att_l  = nn.Parameter(torch.empty(num_heads, output_dim))
        self.att_r  = nn.Parameter(torch.empty(num_heads, output_dim))

        nn.init.xavier_uniform_(self.weight)
        nn.init.xavier_uniform_(self.att_l.unsqueeze(-1))
        nn.init.xavier_uniform_(self.att_r.unsqueeze(-1))

    def forward(self, node_feats, edges):
        total_nodes = node_feats.size(0)
        src_idx, dst_idx = edges

        # projection step
        projected = torch.einsum("hio,ni->hno", self.weight, node_feats)

        src_feat = projected[:, src_idx]   # (H, E, d)
        dst_feat = projected[:, dst_idx]

        # compute attention logits
        att_scores = (
            torch.einsum("hed,hd->he", src_feat, self.att_l) +
            torch.einsum("hed,hd->he", dst_feat, self.att_r)
        )

        att_scores = F.leaky_relu(att_scores, negative_slope=0.2)

        # stable softmax over neighbors
        max_buffer = torch.full((self.num_heads, total_nodes), -float("inf"), device=node_feats.device)
        max_buffer.scatter_reduce_(
            1,
            dst_idx.unsqueeze(0).expand(self.num_heads, -1),
            att_scores,
            reduce="amax",
            include_self=True
        )

        exp_scores = torch.exp(att_scores - max_buffer[:, dst_idx])

        denom = torch.zeros(self.num_heads, total_nodes, device=node_feats.device)
        denom.scatter_add_(
            1,
            dst_idx.unsqueeze(0).expand(self.num_heads, -1),
            exp_scores
        )

        attention = exp_scores / (denom[:, dst_idx] + 1e-16)
        attention = F.dropout(attention, p=self.drop_p, training=self.training)

        # aggregation
        output = torch.zeros(self.num_heads, total_nodes, self.out_dim, device=node_feats.device)

        output.scatter_add_(
            1,
            dst_idx.view(1, -1, 1).expand(self.num_heads, -1, self.out_dim),
            attention.unsqueeze(-1) * src_feat
        )

        # merge heads
        if self.merge:
            return output.permute(1, 0, 2).reshape(total_nodes, self.num_heads * self.out_dim)
        else:
            return output.mean(dim=0)

In [ ]:
class GraphAttentionNet(nn.Module):
    def __init__(self, in_features, hidden_dim, num_classes, heads=4, drop_rate=0.5):
        super().__init__()

        self.attn1 = AttentionLayer(in_features, hidden_dim, heads, drop_rate, merge_heads=True)
        self.attn2 = AttentionLayer(hidden_dim * heads, hidden_dim, heads, drop_rate, merge_heads=True)
        self.attn3 = AttentionLayer(hidden_dim * heads, num_classes, 1, drop_rate, merge_heads=False)

        self.norm1 = nn.BatchNorm1d(hidden_dim * heads)
        self.norm2 = nn.BatchNorm1d(hidden_dim * heads)

        self.residual_proj = nn.Linear(in_features, hidden_dim * heads, bias=False)
        self.dropout_rate  = drop_rate

    def forward(self, x, edge_idx):
        h = F.dropout(x, p=self.dropout_rate, training=self.training)

        # First block
        h_first = self.attn1(h, edge_idx)
        h_first = F.normalize(h_first, p=2, dim=1)
        h = F.elu(self.norm1(h_first)) + self.residual_proj(x)

        # Second block
        h = F.dropout(h, p=self.dropout_rate, training=self.training)
        h_second = self.attn2(h, edge_idx)
        h_second = F.normalize(h_second, p=2, dim=1)
        h = F.elu(self.norm2(h_second)) + h

        # Output layer
        h = F.dropout(h, p=self.dropout_rate, training=self.training)
        logits = self.attn3(h, edge_idx)

        return logits

# GraphSAGE Network

In [ ]:
# GraphSAGE Network
class SageNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, drop_prob=0.5):
        super().__init__()

        # GraphSAGE convolution layers
        self.layer_1 = SAGEConv(input_dim,  hidden_dim, aggr="max")
        self.layer_2 = SAGEConv(hidden_dim, hidden_dim, aggr="max")
        self.layer_3 = SAGEConv(hidden_dim, output_dim, aggr="max")

        # Normalization layers
        self.norm_1 = nn.BatchNorm1d(hidden_dim)
        self.norm_2 = nn.BatchNorm1d(hidden_dim)

        self.dropout_rate = drop_prob

    def forward(self, node_features, edge_idx):
        # First layer
        h = self.layer_1(node_features, edge_idx)
        h = self.norm_1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout_rate, training=self.training)

        # Second layer with skip connection
        h_res = self.layer_2(h, edge_idx)
        h_res = self.norm_2(h_res)
        h_res = F.relu(h_res)

        h = h + h_res   # residual merge
        h = F.dropout(h, p=self.dropout_rate, training=self.training)

        # Output layer (logits)
        logits = self.layer_3(h, edge_idx)
        return logits

**Traning Utilities -> Edge Dropout, Train step, Evaluation**

In [ ]:
# Edge Drop Augmentation
def apply_edge_dropout(edges, drop_prob=0.10):
    """Applies random edge removal for regularization."""
    if drop_prob <= 0:
        return edges

    random_mask = torch.rand(edges.size(1), device=edges.device) > drop_prob

    # ensure at least one edge remains
    return edges[:, random_mask] if random_mask.any() else edges


# Single Training Step
def run_training_step(net, graph, opt, loss_fn, mode):
    net.train()
    opt.zero_grad()

    # dynamic edge sampling
    sampled_edges = apply_edge_dropout(graph.edge_index)

    if mode == "gcn":
        # recompute normalization for modified graph
        ei_new, norm_new = compute_gcn_factors(graph.num_nodes, sampled_edges, compute_device)
        predictions = net(graph.x, ei_new, norm_new)
    else:
        predictions = net(graph.x, sampled_edges)

    loss_value = loss_fn(predictions[graph.train_mask], graph.y[graph.train_mask])
    loss_value.backward()

    torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
    opt.step()

    return loss_value.item()


# Evaluation
@torch.no_grad()
def run_evaluation(net, graph, mask, loss_fn, mode):
    """Runs validation/test evaluation using full graph."""
    net.eval()

    if mode == "gcn":
        logits = net(graph.x, gcn_edges, gcn_weights)
    else:
        logits = net(graph.x, graph.edge_index)

    loss_val = loss_fn(logits[mask], graph.y[mask]).item()

    preds = logits[mask].argmax(dim=1).cpu().numpy()
    labels = graph.y[mask].cpu().numpy()

    precision_val = prec(labels, preds, average="weighted", zero_division=0)
    recall_val    = rec(labels, preds, average="weighted", zero_division=0)

    return loss_val, precision_val, recall_val

**Traning loop with early stopping and LR schedulling**

In [ ]:
# Training Loop
def train_model(model_label, net, graph, mode,
                max_epochs=400, learning_rate=5e-3,
                weight_decay=5e-4, early_stop=40):

    loss_function = nn.CrossEntropyLoss(
        weight=cw_tensor,
        label_smoothing=0.05
    )

    optimizer = torch.optim.Adam(net.parameters(), lr=learning_rate, weight_decay=weight_decay)

    lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=15
    )

    best_loss = float("inf")
    best_weights = None
    wait_counter = 0

    print("\n" + "=" * 56)
    print(f"  Running Training: {model_label}")
    print("=" * 56)

    for ep in range(1, max_epochs + 1):
        train_loss = run_training_step(net, graph, optimizer, loss_function, mode)
        val_loss, _, _ = run_evaluation(net, graph, graph.val_mask, loss_function, mode)

        lr_scheduler.step(val_loss)

        # track best model
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = {k: v.clone() for k, v in net.state_dict().items()}
            wait_counter = 0
        else:
            wait_counter += 1

        # logging
        if ep % 50 == 0:
            diff = train_loss - val_loss
            warning = " overfit" if diff > 0.3 else ""
            print(f"  Epoch {ep:3d} | Train {train_loss:.4f} | "
                  f"Val {val_loss:.4f} | Gap {diff:+.4f} {warning}")

        # early stopping
        if wait_counter >= early_stop:
            print(f"  Stopping early at epoch {ep}")
            break

    net.load_state_dict(best_weights)
    print(f"  Best validation loss: {best_loss:.4f}")

    return net


print("Training pipeline ready")

Training pipeline ready


In [ ]:
#  Model Dimensions
input_dim   = graph_data.num_node_features
hidden_dim  = 64
num_classes = int(graph_data.y.max().item()) + 1

# Initialize Models
model_gcn = NodeGCN(
    input_dim,
    hidden_dim,
    num_classes,
    drop_prob=0.5
).to(compute_device)

model_gat = GraphAttentionNet(
    input_dim,
    hidden_dim // 4,
    num_classes,
    heads=4,
    drop_rate=0.5
).to(compute_device)

model_sage = SageNet(
    input_dim,
    hidden_dim,
    num_classes,
    drop_prob=0.5
).to(compute_device)


# Training Phase
model_gcn = train_model(
    "GCN (manual implementation)",
    model_gcn,
    graph_data,
    mode="gcn"
)

model_gat = train_model(
    "GAT (manual implementation)",
    model_gat,
    graph_data,
    mode="gat"
)

model_sage = train_model(
    "GraphSAGE ",
    model_sage,
    graph_data,
    mode="sage"
)

print("\nTraining complete for all architectures ")


  Running Training: GCN (manual implementation)
  Epoch  50 | Train 0.2953 | Val 0.2705 | Gap +0.0248 
  Stopping early at epoch 95
  Best validation loss: 0.2703

  Running Training: GAT (manual implementation)
  Epoch  50 | Train 0.2859 | Val 0.1441 | Gap +0.1418 
  Stopping early at epoch 97
  Best validation loss: 0.1376

  Running Training: GraphSAGE 
  Epoch  50 | Train 0.1356 | Val 0.1289 | Gap +0.0067 
  Stopping early at epoch 65
  Best validation loss: 0.1205

Training complete for all architectures 


# Summarry of all model

In [ ]:
eval_loss_fn = nn.CrossEntropyLoss(
    weight=cw_tensor,
    label_smoothing=0.05
)

@torch.no_grad()
def compute_metrics(net, graph, mask, loss_fn, mode):
    net.eval()

    if mode == "gcn":
        logits = net(graph.x, gcn_edges, gcn_weights)
    else:
        logits = net(graph.x, graph.edge_index)

    loss_val = loss_fn(logits[mask], graph.y[mask]).item()

    predictions = logits[mask].argmax(dim=1).cpu().numpy()
    targets     = graph.y[mask].cpu().numpy()

    acc  = (predictions == targets).mean()
    prec_val = prec(targets, predictions, average="weighted", zero_division=0)
    rec_val  = rec(targets, predictions, average="weighted", zero_division=0)

    from sklearn.metrics import f1_score
    f1_val = f1_score(targets, predictions, average="weighted", zero_division=0)

    return loss_val, acc, prec_val, rec_val, f1_val


#  Run Evaluation
_, acc_gcn,  prec_gcn,  rec_gcn,  f1_gcn  = compute_metrics(model_gcn,  graph_data, graph_data.test_mask, eval_loss_fn, "gcn")
_, acc_gat,  prec_gat,  rec_gat,  f1_gat  = compute_metrics(model_gat,  graph_data, graph_data.test_mask, eval_loss_fn, "gat")
_, acc_sage, prec_sage, rec_sage, f1_sage = compute_metrics(model_sage, graph_data, graph_data.test_mask, eval_loss_fn, "sage")


results_table = [
    ("GCN",        acc_gcn,  prec_gcn,  rec_gcn,  f1_gcn),
    ("GAT",     acc_gat,  prec_gat,  rec_gat,  f1_gat),
    ("GraphSAGE (PyG)",     acc_sage, prec_sage, rec_sage, f1_sage),
]

print("\n" + "=" * 75)
print(f"{'Model':<25} {'Acc':>8} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 75)

for name, acc, p, r, f1 in results_table:
    print(f"{name:<25} {acc:>8.4f} {p:>10.4f} {r:>10.4f} {f1:>10.4f}")

print("=" * 75)


Model                          Acc  Precision     Recall         F1
---------------------------------------------------------------------------
GCN                         0.9850     0.9852     0.9850     0.9850
GAT                         0.9975     0.9975     0.9975     0.9975
GraphSAGE (PyG)             1.0000     1.0000     1.0000     1.0000
